# KV-Hadamard — Kaggle (free T4) validation

**Author: Aryan Gupta** · repo: github.com/aryxnsdfs/kv-hadamard

Kaggle's free GPUs (T4/P100) are pre-Ampere, so **flash-attn 2 won't run** on them. KIVI calls flash only inside `_flash_attention_forward`, so we skip flash-attn entirely and **monkeypatch that method with an SDPA prefill** (works on T4). Everything else — the KIVI INT4 K kernel, the B2 packed 2-bit V — runs fine on Turing.

**Before running (two Kaggle gotchas):**
1. Notebook settings → **Accelerator: GPU T4 x2**
2. Notebook settings → **Internet: ON** (off by default — pip/HF downloads fail silently otherwise)

Tests: (A) fp16 vs KIVI vs B2 quality/memory/speed on Llama-2-7B; (B) long-context KV-memory scaling 4k→32k where the compression win becomes dramatic. Outputs land in `/kaggle/working/`.

## 0. Environment check — GPU + internet

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, urllib.request
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpus', torch.cuda.device_count())
try:
    urllib.request.urlopen('https://huggingface.co', timeout=5); print('INTERNET OK')
except Exception as e:
    print('INTERNET OFF -> enable it in notebook settings, else nothing downloads:', e)

## 1. gcc-12 (CUDA nvcc rejects gcc>12) + clone repos

In [ ]:
import os
!sudo apt-get update -qq && sudo apt-get install -y -qq gcc-12 g++-12
os.environ['CC']='gcc-12'; os.environ['CXX']='g++-12'; os.environ['CUDAHOSTCXX']='g++-12'
%cd /kaggle/working
![ -d KIVI ] || git clone -q https://github.com/jy-yuan/KIVI.git
![ -d kv-hadamard ] || git clone -q https://github.com/aryxnsdfs/kv-hadamard.git
print('ready')

## 2. Deps — pinned combo, NO flash-attn (we patch it out)

In [ ]:
!pip install -q ninja packaging
!pip install -q transformers==4.36.2 accelerate==0.25.0 bitsandbytes==0.43.0
!pip install -q datasets sentencepiece protobuf matplotlib
print('deps installed (flash-attn intentionally skipped)')

## 3. Build KIVI's CUDA quant kernels (compiles for T4 SM75)

In [ ]:
!cd /kaggle/working/KIVI/quant && pip install -e . --no-build-isolation 2>&1 | tail -6
print('KIVI kernels built')

## 4. Imports + the SDPA prefill patch (replaces flash on Turing)

In [ ]:
import sys, types, math
sys.path.insert(0, '/kaggle/working/KIVI')
sys.path.insert(0, '/kaggle/working/kv-hadamard')
%cd /kaggle/working/kv-hadamard
import torch, torch.nn.functional as F
from transformers.models.llama.modeling_llama import repeat_kv
from models.llama_kivi import LlamaForCausalLM_KIVI  # imports WITHOUT flash (it's lazy)
from b2_packed_int2 import patch_packed_int2_v
from phase2_kivi_harness import load_kivi, CONFIG
from fp16_baseline_harness import measure_kv_memory, measure_decode_tps, free_cuda, _analytic_kv_mb
from probe_cache_ppl import ppl_cache, PREFILL

def _sdpa_prefill(self, query_states, key_states, value_states, attention_mask=None,
                  query_length=None, dropout=0.0, **kw):
    """Drop-in for KIVI's _flash_attention_forward. Inputs [b,n,h,d]; causal GQA
    SDPA (math/mem-efficient backend works on Turing). Returns [b,n,h,d]."""
    q = query_states.transpose(1, 2)
    k = repeat_kv(key_states.transpose(1, 2), self.num_key_value_groups)
    v = repeat_kv(value_states.transpose(1, 2), self.num_key_value_groups)
    o = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
    return o.transpose(1, 2).contiguous()

def patch_no_flash(model):
    n = 0
    for m in model.modules():
        if hasattr(m, 'k_bits') and hasattr(m, '_flash_attention_forward'):
            m._flash_attention_forward = types.MethodType(_sdpa_prefill, m); n += 1
    return n

print('imports + SDPA patch ready')

## 5. Data helper

In [ ]:
from datasets import load_dataset
def passage(tok, N, offset=0):
    ds = load_dataset(CONFIG['wikitext_dataset'], CONFIG['wikitext_config'], split=CONFIG['wikitext_split'])
    ids = tok('\n\n'.join(ds['text']), return_tensors='pt').input_ids
    return ids[:, offset:offset+N]

## TEST A — fp16 vs KIVI vs B2 on Llama-2-7B

fp16 loads across both T4s (`device_map=auto`); KIVI/B2 are NF4 (~4GB, single GPU). fp16 KV reported analytic (device-split makes empirical unreliable); KIVI/B2 empirical (real bytes).

In [ ]:
CONFIG['model_name'] = 'NousResearch/Llama-2-7b-hf'
results_A = {}

# fp16 across 2x T4 (native SDPA attn -> Turing-safe, no flash)
print('=== fp16 (dual-T4) ===', flush=True)
from transformers import LlamaForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(CONFIG['model_name'])
if tok.pad_token is None: tok.pad_token = tok.eos_token
m = LlamaForCausalLM.from_pretrained(CONFIG['model_name'], torch_dtype=torch.float16,
                                     device_map='auto', attn_implementation='sdpa').eval()
ids = passage(tok, 896).to('cuda:0')
results_A['fp16'] = {'ppl': ppl_cache(m, ids), 'kv_mb': _analytic_kv_mb(2048, model=m), 'tps': None}
print('fp16', results_A['fp16'], flush=True)
del m; free_cuda()

In [ ]:
# KIVI + B2 (NF4, single GPU, SDPA prefill patch)
print('=== kivi ===', flush=True)
model, tok, _ = load_kivi(); patch_no_flash(model)
ids = passage(tok, 896).to(CONFIG['device'])
results_A['kivi'] = {'ppl': ppl_cache(model, ids), 'kv_mb': measure_kv_memory(model, tok, 2048)['empirical_mb'], 'tps': measure_decode_tps(model, tok)['tokens_per_sec']}
print('kivi', results_A['kivi'], flush=True); free_cuda()

print('=== B2 packed Hadamard-INT2 ===', flush=True)
patch_packed_int2_v(model); patch_no_flash(model)  # re-apply prefill patch after B2 swap
results_A['b2'] = {'ppl': ppl_cache(model, ids), 'kv_mb': measure_kv_memory(model, tok, 2048)['empirical_mb'], 'tps': measure_decode_tps(model, tok)['tokens_per_sec']}
print('b2', results_A['b2'], flush=True)
del model; free_cuda()

print('\n=== TEST A SUMMARY (Llama-2-7B) ===')
print(f"{'':<8}{'PPL':>10}{'KV MB@2048':>13}{'tok/s':>9}")
for k in ['fp16','kivi','b2']:
    r=results_A[k]; t=f"{r['tps']:.1f}" if r['tps'] else '-(split)'
    print(f"{k:<8}{r['ppl']:>10.4f}{r['kv_mb']:>13.1f}{t:>9}")

## TEST B — long-context KV-memory scaling (the dramatic win)

`togethercomputer/LLaMA-2-7B-32K` (Llama-2 arch, 32k window). NF4 weights (~4GB) leave the T4's remaining ~11GB for KV, so KIVI and B2 reach long contexts; fp16 KV blows up. Real measured bytes, kivi vs B2 (fp16 analytic for reference).

In [ ]:
CONFIG['model_name'] = 'togethercomputer/LLaMA-2-7B-32K'
LENGTHS = [4096, 8192, 16384, 32768]
results_B = {'lengths': LENGTHS, 'fp16_analytic': {}}

def kv_scan(tag, patch=None):
    print(f'=== {tag} KV scan ===', flush=True)
    model, tok, _ = load_kivi(); patch_no_flash(model)
    if patch: patch(model); patch_no_flash(model)
    row = {}
    for N in LENGTHS:
        try:
            mb = measure_kv_memory(model, tok, N)['empirical_mb']; row[N] = round(mb,1)
            print(f'  {tag} @{N}: {mb:.1f} MB', flush=True)
        except Exception as e:
            row[N] = 'OOM' if 'memory' in str(e).lower() else f'ERR'; print(f'  {tag} @{N}: {row[N]} ({e})', flush=True); free_cuda()
        # fp16 analytic reference (architecture closed-form, device-independent)
        results_B['fp16_analytic'][N] = round(_analytic_kv_mb(N, model=model), 1)
    del model; free_cuda(); return row

results_B['kivi'] = kv_scan('kivi')
results_B['b2']   = kv_scan('b2', patch=patch_packed_int2_v)
print('\nTEST B:', results_B)

## Plots + save to /kaggle/working

In [ ]:
import json, matplotlib.pyplot as plt
json.dump({'A': results_A, 'B': results_B}, open('/kaggle/working/kaggle_results.json','w'), indent=2)

fig, axs = plt.subplots(1, 2, figsize=(12,4.5))
ks=['fp16','kivi','b2']; cols=['#8a8a8a','#444','#1a7f3c']
axs[0].bar(ks, [results_A[k]['ppl'] for k in ks], color=cols, edgecolor='k'); axs[0].set_title('Test A — cache-path PPL (7B)')
axs[0].set_ylim(min(results_A[k]['ppl'] for k in ks)-0.05, max(results_A[k]['ppl'] for k in ks)+0.05)
axs[1].bar(ks, [results_A[k]['kv_mb'] for k in ks], color=cols, edgecolor='k'); axs[1].set_title('Test A — KV MB @2048')
fig.tight_layout(); fig.savefig('/kaggle/working/kaggle_testA.png'); plt.show()

fig, ax = plt.subplots(figsize=(8,5))
for k,c in [('fp16_analytic','#8a8a8a'),('kivi','#444'),('b2','#1a7f3c')]:
    d=results_B[k]; xs=[N for N in LENGTHS if isinstance(d.get(N),(int,float))]
    if xs: ax.plot([x//1024 for x in xs],[d[N] for N in xs],'o-',label=k,color=c,lw=2)
ax.set_xlabel('context (k tokens)'); ax.set_ylabel('KV cache MB'); ax.set_title('Test B — KV memory scaling (missing = OOM)')
ax.legend(); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig('/kaggle/working/kaggle_testB.png'); plt.show()
print('saved kaggle_results.json + 2 PNGs to /kaggle/working (grab them from the Output tab)')